# Exploring the Amazon Fashion metadata

Quick look at the data before building anything. Questions i wanted answered:

1. which fields are actually populated (the pdf lists 14 fields, how many are usable?)
2. does metadata quality depend on popularity (rating count)? this decides how the demo subset gets picked
3. which local embedding model is good enough for conversational queries, and does bm25 add anything
4. what does the LLM planner produce for a few human style queries

Run from the repo root with the raw file in `data/raw/` (`make data`).

In [ ]:
import gzip, json, re, itertools, collections, time, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, "src")
from stylist.catalog import parse_price, derive_audience, group_key

RAW = Path("data/raw/meta_Amazon_Fashion.jsonl.gz")
assert RAW.exists(), "run `make data` first"
plt.rcParams["figure.dpi"] = 110

## 1. Field coverage

Stream the whole file once (826K rows, about a minute) and count what is non-empty.

In [ ]:
keys = collections.Counter(); nonempty = collections.Counter(); n = 0
rating_number = []; title_len = []; dept = collections.Counter(); cat_depth = collections.Counter()
prices = []; years = collections.Counter()
with gzip.open(RAW, "rt") as f:
    for line in f:
        r = json.loads(line); n += 1
        for k, v in r.items():
            keys[k] += 1
            if v not in (None, "", [], {}): nonempty[k] += 1
        rating_number.append(r.get("rating_number") or 0)
        title_len.append(len(r.get("title") or ""))
        cat_depth[len(r.get("categories") or [])] += 1
        d = r.get("details") or {}
        if "Department" in d: dept[d["Department"].strip().lower()] += 1
        p, status = parse_price(r.get("price"))
        if p is not None: prices.append(p)
        m = re.search(r"(\d{4})", d.get("Date First Available", ""))
        if m: years[m.group(1)] += 1
print("rows:", n)
cov = pd.Series({k: nonempty[k] / n for k in keys}).sort_values(ascending=False)
cov.round(3).to_frame("non-empty share")

So: `categories` is empty for every single row and `bought_together` is always null. `price` exists for 6%, `description` for 7%. The title is the one field that is always there, and it is dense (brand, gender, product type, colour, size all crammed in). That shaped the whole design: the title carries retrieval, everything else is a bonus.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].hist(np.clip(title_len, 0, 250), bins=50); ax[0].set_title("title length (chars)")
rn = np.array(rating_number); ax[1].hist(np.log10(rn + 1), bins=40); ax[1].set_title("log10(rating_number + 1)")
ax[2].hist(np.clip(prices, 0, 150), bins=50); ax[2].set_title(f"price (USD, n={len(prices):,})")
plt.tight_layout(); plt.show()
print("rating_number: median", np.median(rn), " >=5:", (rn >= 5).sum(), " >=20:", (rn >= 20).sum(), " >=100:", (rn >= 100).sum())
print("price median", np.median(prices), "p90", np.percentile(prices, 90))
print("categories depth:", dict(cat_depth))
print("department values (top):", dept.most_common(8))
print("years listed (top):", sorted(years.items())[-8:])

## 2. Does popularity buy better metadata?

The ingest step records coverage per rating bucket (`data/processed/ingest_stats.json`, written by `make ingest`). This is the evidence behind indexing the most-rated 100K items by default for the demo.

In [ ]:
stats = json.loads(Path("data/processed/ingest_stats.json").read_text())
by_bucket = pd.DataFrame(stats["by_rating_bucket"]).T
by_bucket.index.name = "rating_number bucket"
display(by_bucket)
by_bucket.drop(columns=["rows", "image_url"]).plot.bar(figsize=(9, 3.4), title="share of rows with the field populated")
plt.ylabel("share"); plt.tight_layout(); plt.show()

Price coverage goes from 5% (0-4 ratings) to 26% (100+ ratings), features from 50% to 79%, department from 12% to 49%. Images are 100% everywhere. Popular listings are simply better documented, which matters a lot for a demo where the LLM has to explain its picks. The trade-off (long tail products never get indexed) is measured in `docs/evaluation.md` by also running the full catalog.

## 3. Audience heuristic and variant grouping

No taxonomy, so audience is guessed from `details.Department` first and the title second. Variant listings (same product, other size or colour) get a `group_key` so they collapse at query time instead of being deleted at ingest.

In [ ]:
sample = []
with gzip.open(RAW, "rt") as f:
    for line in itertools.islice(f, 0, 300000, 12000):
        sample.append(json.loads(line))
rows = [{"audience": derive_audience(r["title"], (r.get("details") or {}).get("Department")),
         "group_key": group_key(r["title"])[:60], "title": r["title"][:70]} for r in sample]
pd.DataFrame(rows).head(25)

## 4. Embedding models vs BM25 on conversational queries

40K listings with at least 5 ratings, 8 queries written the way a person would type them. No labels here, just eyeballing the top 3 per model. The full comparison is slow on CPU, takes ~2 min on an M-series GPU.

In [ ]:
from sentence_transformers import SentenceTransformer
import bm25s
from stylist.catalog import build_doc_text

docs, titles = [], []
with gzip.open(RAW, "rt") as f:
    for line in f:
        r = json.loads(line)
        if (r.get("rating_number") or 0) < 5: continue
        docs.append(build_doc_text(r)); titles.append(r["title"][:80])
        if len(docs) >= 40000: break
queries = ["I need an outfit to go to the beach this summer", "warm waterproof boots for hiking in snow",
           "elegant black dress for a wedding guest", "men's slim fit chinos for the office",
           "cozy oversized sweater for fall", "running shoes with good arch support for flat feet",
           "something to keep my ears warm in winter", "comfortable sandals for walking around europe"]

bm = bm25s.BM25(); bm.index(bm25s.tokenize(docs, stopwords="en", show_progress=False), show_progress=False)
results = {"bm25": []}
for q in queries:
    sc = bm.get_scores(bm25s.tokenize([q], stopwords="en", return_ids=False, show_progress=False)[0])
    results["bm25"].append([titles[i] for i in np.argsort(-sc)[:3]])

models = {"bge-small-en-v1.5": ("BAAI/bge-small-en-v1.5", "Represent this sentence for searching relevant passages: "),
          "all-MiniLM-L6-v2": ("sentence-transformers/all-MiniLM-L6-v2", ""),
          "arctic-embed-xs": ("Snowflake/snowflake-arctic-embed-xs", "Represent this sentence for searching relevant passages: ")}
speed = {}
for name, (path, prefix) in models.items():
    m = SentenceTransformer(path)
    t = time.time(); E = m.encode(docs, batch_size=128, normalize_embeddings=True, show_progress_bar=False); speed[name] = len(docs) / (time.time() - t)
    Q = m.encode([prefix + q for q in queries], normalize_embeddings=True)
    S = Q @ E.T
    results[name] = [[titles[i] for i in np.argsort(-S[qi])[:3]] for qi in range(len(queries))]
print({k: f"{v:.0f} docs/s" for k, v in speed.items()})
for qi, q in enumerate(queries):
    print("\n###", q)
    for name in results:
        print(f"  {name:18s}", " | ".join(t[:45] for t in results[name][qi]))

What i took from this:

* bm25 alone is hopeless for "outfit for the beach" (it matches *outfit* and *summer* in toddler listings) and for "keep my ears warm", but it is great at exact phrases like "wedding guest".
* bge-small gives the most sensible lists of the three and is still ~1100 docs/s on the laptop GPU, so it became the default. arctic-xs is faster and close behind, MiniLM is noticeably weaker.
* gender leaks through on dense only ("men's chinos" returned a women's office suit at rank 3), which is why the service has an audience filter applied as a mask before ranking.

Hence hybrid: dense + bm25 fused with reciprocal rank fusion.

## 5. What the planner does with human queries

Needs an LLM key in the environment (`LLM_PROVIDER`, see `.env.example`). Skipped otherwise.

In [ ]:
import asyncio
from stylist.config import Settings
from stylist.llm import make_llm_client
from stylist.planner import LLMPlanner

settings = Settings.from_env()
llm = make_llm_client(settings)
if llm is None:
    print("no LLM configured, skipping")
else:
    print("model:", llm.model)
    planner = LLMPlanner(llm)
    for q in ["I need an outfit to go to the beach this summer",
              "what should my husband wear to an outdoor wedding in june, budget 200 total",
              "chaussures de running pas cheres pour femme"]:
        plan = asyncio.run(planner.plan(q, timeout=60))
        print("\n###", q)
        print("  intent:", plan.intent, "| audience:", plan.audience, "| budget:", plan.budget_max, plan.budget_scope)
        for s in plan.slots:
            print(f"  - {s.name:14s} q={s.search_query!r}  kw={s.keywords}  budget={s.budget_max}")

The planner translates the french query, splits the outfit into pieces and allocates a total budget across slots (shoes get more than the tie). Those per-slot queries are what the retriever actually searches, the shopper's sentence never hits the index directly.